# Create MS Society (UK) Awards (GRANT PATTERN, Europe PMC grant deposit + funder website)

**Awarding body:** Multiple Sclerosis Society of Great Britain and Northern Ireland,
F4320320084 (GB). NOT the US National MS Society (F4320306161), MS Society of Canada
(F4320319986) or MS Trust (F4320320101).

**Sources (both the funder's own publications, merged in the script):**
1. The MS Society's grant list as deposited with Europe PMC (it is a Europe PMC funder),
   GRIST API `ga:"Multiple Sclerosis Society"`: grant number, title, abstract, award type,
   stream, dates, GBP amount, holder + ORCID, institution + ROR. 307 grants 1999-2020
   (the deposit stops at 2020). GRIST paging is unstable; the script unions full passes
   over several query spellings until the distinct-record count reaches HitCount.
2. mssociety.org.uk "Search our research projects" (182 pages: lead researcher,
   institution, MS Society funding, status; no grant number, no dates). 73 pages match a
   GRIST grant (same PI + amount within GBP 2 or 3%, or a unique non-round amount) and only
   supply the landing page. 72 active/approved/new pages with no GRIST match are added as
   website-only awards (mostly 2021+ grants, no dates). 37 unmatched COMPLETED pages are
   skipped: they predate 2021 and are more likely GRIST grants with a drifted amount/PI, or
   trial-results pages, than new grants.
No 360Giving file (registry checked 2026-09-30). All research money (no non-research grants).
**379 awards; 100% title/amount (GBP)/PI/institution, 79% dates, 47% abstract, 74% ROR, 23% ORCID.**

**`funder_award_id` (§2.1.1):** checked against the citation side
(`works?filter=grants.funder:F4320320084` / `crossref_work.grants` + `europepmc_work_funders`
stubs). Old-scheme grants (numbers 491-996, 1999-2012) are cited `NNN/YY` (YY = award
round, not derivable from dates): the script crosswalks GRIST numbers to the exact cited
`NNN/YY` string of an existing F4320320084 award stub (48 grants), else ships the bare MS
Society grant number (259; new-scheme 7-130 are cited as "Grant 76", "Ref 71" etc., no
consistent form). Website-only awards get a synthetic `MSSOC-WEB-<Drupal node id>`.

**Prerequisites:**
- Run `scripts/local/ms_society_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/ms_society/ms_society_projects.parquet` (§1.4 shrink guard).

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320320084`
- display_name / ror_id / doi: read from `openalex.funders.funders` by ID (exactly one row)

**Priority:** `517` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).
**Provenance:** `ms_society_grants`.

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ms_society_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/ms_society/ms_society_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_grants, COUNT(DISTINCT funder_award_id) as distinct_ids FROM openalex.awards.ms_society_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.ms_society_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320320084 is a Crossref-registered funder, so it MUST resolve to exactly 1 row in
`openalex.funders.funders`. The assert fails the run otherwise.

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320320084 must resolve to exactly one openalex.funders.funders row')
FROM openalex.funders.funders WHERE funder_id = 4320320084;

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders WHERE funder_id = 4320320084;

## Step 2: Create MS Society (UK) Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ms_society_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320320084  -- MS Society (UK)
),

people AS (
    -- Grant holders in source order -> investigator structs.
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        funder_award_id,
        TRANSFORM(
            FILTER(
                from_json(people_json, 'ARRAY<STRUCT<given_name:STRING, family_name:STRING, title:STRING, orcid:STRING, institution:STRING, institution_ror:STRING>>'),
                p -> p.family_name IS NOT NULL
            ),
            p -> named_struct(
                'given_name', NULLIF(TRIM(p.given_name), ''),
                'family_name', NULLIF(TRIM(p.family_name), ''),
                'orcid', CASE WHEN p.orcid RLIKE '^[0-9]{4}-[0-9]{4}-[0-9]{4}-[0-9]{3}[0-9X]$' THEN CONCAT('https://orcid.org/', p.orcid) END,
                'role_start', CAST(NULL AS DATE),
                'affiliation', named_struct(
                    'name', NULLIF(TRIM(p.institution), ''),
                    'country', CAST(NULL AS STRING),
                    'ids', CASE WHEN p.institution_ror IS NOT NULL
                                THEN array(named_struct('id', p.institution_ror, 'type', 'ror', 'asserted_by', 'funder'))
                                ELSE CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) END
                )
            )
        ) AS investigators
    FROM openalex.awards.ms_society_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.abstract), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN NULLIF(TRIM(g.currency), '') END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    g.funding_type as funding_type,
    NULLIF(TRIM(g.award_type), '') as funder_scheme,
    'ms_society_grants' as provenance,
    TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
    TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
    YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
    YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
    -- try_element_at: safe on empty arrays (§2.3)
    try_element_at(p.investigators, 1) as lead_investigator,
    try_element_at(p.investigators, 2) as co_lead_investigator,
    CASE WHEN size(p.investigators) > 0 THEN p.investigators END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.ms_society_raw g
JOIN people p ON p.funder_award_id = g.funder_award_id
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- §2.1.1 shape check: every funder_award_id is the citable reference, no registry/360G ids.
SELECT assert_true(
    SUM(CASE WHEN funder_award_id RLIKE '^([0-9]{1,3}(/[0-9]{2})?|MSSOC-WEB-[0-9]+)$' THEN 0 ELSE 1 END) = 0
    AND SUM(CASE WHEN funder_award_id LIKE '360G%' THEN 1 ELSE 0 END) = 0
    AND COUNT(*) = COUNT(DISTINCT id),
    'funder_award_id shape / uniqueness check failed')
FROM openalex.awards.ms_society_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ms_society_grants' AND priority = 517;

-- Priority 517: direct-from-funder ingest. Higher wins under the 2026-06-20
-- DESC dedup (oxjob #500), so it outranks the crossref/europepmc citation stubs
-- (priority 0/1) that share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    517 as priority
FROM openalex.awards.ms_society_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids FROM openalex.awards.ms_society_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, currency, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.orcid,
       lead_investigator.affiliation.name, size(investigators) as n_inv
FROM openalex.awards.ms_society_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_amount
FROM openalex.awards.ms_society_awards GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_amount
FROM openalex.awards.ms_society_awards WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.family_name AS family, lead_investigator.given_name AS given, COUNT(*) AS n
FROM openalex.awards.ms_society_awards GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.orcid) as has_pi_orcid,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.ms_society_awards;

In [ ]:
%sql
-- Overlap with existing citation stubs for this funder (these collapse in CreateAwards dedup).
SELECT c.provenance, COUNT(*) as stub_rows, COUNT(t.id) as matched_by_this_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.ms_society_awards t ON t.id = c.id
WHERE c.funder_id = 4320320084 AND c.priority < 517
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ms_society_grants'
GROUP BY provenance, priority;